In [6]:
!pip install pandas openpyxl
!pip install python-docx


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 75.7 MB/s eta 0:00:00


In [29]:
import pandas as pd

# Load all sheets
file_path = "Patients_for_SHEPHERD_genes.xlsx"
all_sheets = pd.read_excel(file_path, sheet_name=None)  # read all sheets

# Create dictionary: key = patient ID (sheet name), value = list of genes
patient_genes = {}
for patient_id, df in all_sheets.items():
    genes = []
    for cell in df.iloc[:, 0]:
        # Split by comma and strip whitespace
        split_genes = [g.strip() for g in str(cell).split(",")]
        genes.extend(split_genes)
    patient_genes[patient_id] = genes

# Print first few entries
for patient_id, genes in patient_genes.items():
    print(f"{patient_id}: {genes[:5]} ...")  # show first 5 genes for brevity


BF028: ['ABCA4', 'ACE2', 'AGBL4', 'AGBL4-IT1', 'AHNAK2'] ...
BF029: ['ACTL6B', 'ADGRB3', 'ADORA2B', 'AHNAK2', 'ALDH18A1'] ...
BF037: ['ANKRD20A3P', 'ANKRD36', 'ARHGEF34P', 'ASH1L', 'ATIC'] ...
BF042: ['ADGRB1', 'AFAP1L2', 'AHDC1', 'AHNAK2', 'ALPK2'] ...
BF045: ['ABCC8', 'ACOX2', 'ADAMTSL4', 'AHNAK', 'ANKLE1'] ...
BF050: ['ABCG2', 'ABHD1', 'ADH1B', 'ADORA2B', 'AGO1'] ...


In [30]:
patient_genes['BF042'][:5]

['ADGRB1', 'AFAP1L2', 'AHDC1', 'AHNAK2', 'ALPK2']

In [31]:
import requests
import time

# Global list to store missing genes
missing_genes_log = []

def get_ensembl_id(gene_symbol):
    """
    Query Ensembl REST API to get Ensembl Gene ID for a given human gene symbol.
    Raises ValueError if not found.
    """
    url = f"https://rest.ensembl.org/xrefs/symbol/homo_sapiens/{gene_symbol}?content-type=application/json"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            for entry in data:
                if entry.get("type") == "gene":
                    return entry.get("id")
            raise ValueError(f"Ensembl ID not found for gene symbol: {gene_symbol}")
        else:
            raise ValueError(f"Failed request for {gene_symbol}: {response.status_code}")
    except Exception as e:
        raise ValueError(f"Error fetching {gene_symbol}: {e}")

def convert_gene_list_to_ensembl(gene_list):
    """
    Convert a list of gene symbols to Ensembl IDs.
    Logs missing genes to `missing_genes_log`.
    """
    ensembl_ids = []
    for i, gene in enumerate(gene_list, 1):
        try:
            ensembl_id = get_ensembl_id(gene)
            ensembl_ids.append(ensembl_id)
            print(f"[{i}/{len(gene_list)}] {gene} → {ensembl_id}")
        except ValueError:
            print(f"⚠️ Skipping {gene}, no Ensembl ID found")
            missing_genes_log.append(gene)
        time.sleep(0.2)  # Respect API rate limit
    return ensembl_ids




In [32]:
# --- Convert patient gene lists to Ensembl IDs ---
for patient_id in patient_genes.keys():
    print(f"Analyzing {patient_id}")
    patient_genes[patient_id] = convert_gene_list_to_ensembl(patient_genes[patient_id])

# Write missing genes to a file
with open("missing_genes.txt", "w") as f:
    for gene in missing_genes_log:
        f.write(gene + "\n")

print("Missing genes saved to 'missing_genes.txt'")

[1/184] ABCA4 → ENSG00000198691
[2/184] ACE2 → ENSG00000130234
[3/184] AGBL4 → ENSG00000186094
[4/184] AGBL4-IT1 → ENSG00000225623
[5/184] AHNAK2 → ENSG00000185567
[6/184] AMY1A → ENSG00000237763
[7/184] AMY1C → ENSG00000187733
[8/184] ANKRD11 → ENSG00000167522
[9/184] ANKRD20A3P → ENSG00000276203
[10/184] ANKRD23 → ENSG00000163126
[11/184] AQP12A → ENSG00000185176
[12/184] ARHGEF34P → ENSG00000204959
[13/184] ASPSCR1 → ENSG00000169696
[14/184] BCL11A → ENSG00000119866
[15/184] BRAT1 → ENSG00000106009
[16/184] C3 → ENSG00000125730
[17/184] C6 → ENSG00000101182
[18/184] CAMTA1 → ENSG00000171735
[19/184] CBS → ENSG00000160200
[20/184] CEACAM16 → ENSG00000213892
[21/184] CEL → ENSG00000170835
[22/184] CFAP410 → ENSG00000160226
[23/184] CFAP43 → ENSG00000197748
[24/184] CFAP45 → ENSG00000213085
[25/184] COL5A1 → ENSG00000130635
[26/184] COL7A1 → ENSG00000114270
[27/184] CRACD → ENSG00000109265
[28/184] CT45A3 → ENSG00000269096
[29/184] CTAGE15 → ENSG00000271079
[30/184] CTAGE4 → ENSG000002

In [35]:
patient_genes

{'BF028': ['ENSG00000198691',
  'ENSG00000130234',
  'ENSG00000186094',
  'ENSG00000225623',
  'ENSG00000185567',
  'ENSG00000237763',
  'ENSG00000187733',
  'ENSG00000167522',
  'ENSG00000276203',
  'ENSG00000163126',
  'ENSG00000185176',
  'ENSG00000204959',
  'ENSG00000169696',
  'ENSG00000119866',
  'ENSG00000106009',
  'ENSG00000125730',
  'ENSG00000101182',
  'ENSG00000171735',
  'ENSG00000160200',
  'ENSG00000213892',
  'ENSG00000170835',
  'ENSG00000160226',
  'ENSG00000197748',
  'ENSG00000213085',
  'ENSG00000130635',
  'ENSG00000114270',
  'ENSG00000109265',
  'ENSG00000269096',
  'ENSG00000271079',
  'ENSG00000213214',
  'ENSG00000271321',
  'ENSG00000179142',
  'ENSG00000165617',
  'ENSG00000157326',
  'ENSG00000151914',
  'ENSG00000183690',
  'ENSG00000283632',
  'ENSG00000285950',
  'ENSG00000285720',
  'ENSG00000285607',
  'ENSG00000285975',
  'ENSG00000144554',
  'ENSG00000275395',
  'ENSG00000143631',
  'ENSG00000225899',
  'ENSG00000205777',
  'ENSG00000236362',
  'E

In [7]:
from docx import Document
import re

# Load the Word document
file_path = "Patients_for_SHEPHERD_phenotypes.docx"
doc = Document(file_path)

patient_hpo = {}
current_patient = None

for para in doc.paragraphs:
    text = para.text.strip()
    if not text:
        continue  # skip empty lines

    # Check if this is a patient ID (assuming it does not start with HP:)
    if not text.startswith("HP:"):
        current_patient = text
        patient_hpo[current_patient] = []
    else:
        # Extract only the HPO term (before any space or hyphen)
        hpo_match = re.match(r"(HP:\d+)", text)
        if hpo_match and current_patient:
            patient_hpo[current_patient].append(hpo_match.group(1))

# Print the dictionary
for patient_id, hpo_terms in patient_hpo.items():
    print(f"{patient_id}: {hpo_terms[:5]} ...")  # first 5 HPO terms


BF028: ['HP:0001511', 'HP:0001531', 'HP:0004322', 'HP:0002007', 'HP:0000369'] ...
BF029: ['HP:0000252', 'HP:0001508', 'HP:0001263', 'HP:0002079', 'HP:0008936'] ...
BF037: ['HP:0001791', 'HP:0002240', 'HP:0002908', 'HP:0001409'] ...
BF042: ['HP:0001319', 'HP:0001250', 'HP:0006989', 'HP:0001999', 'HP:0000347'] ...
BF045: ['HP:0007704', 'HP:0001999', 'HP:0001263', 'HP:0002376', 'HP:0008936'] ...
BF050: ['HP:0009469', 'HP:0000505', 'HP:0008527', 'HP:0002240', 'HP:0002910'] ...


In [13]:
# patient_genes, patient_hpo

In [36]:
# Combine into a list of dicts
import json
combined_list = []
for idx, patient_id in enumerate(patient_genes.keys(), start=1):
    combined_list.append({
        "id": idx,
        "all_candidate_genes": patient_genes[patient_id],
        "positive_phenotypes": patient_hpo.get(patient_id, [])
    })

# Save to JSON file
with open("combined_patient_data.json", "w") as f:
    json.dump(combined_list, f, indent=2)

print("JSON file saved as 'combined_patient_data.json'")

JSON file saved as 'combined_patient_data.json'


In [37]:
combined_list = []
for patient_id in patient_genes.keys():
    combined_list.append({
        "id": patient_id,  # use patient ID instead of numeric index
        "all_candidate_genes": patient_genes[patient_id],
        "positive_phenotypes": patient_hpo.get(patient_id, [])
    })

# Save each dict as a separate line in a text file
with open("combined_patient_data.txt", "w") as f:
    for patient_dict in combined_list:
        json_line = json.dumps(patient_dict)
        f.write(json_line + "\n")

print("Text file saved as 'combined_patient_data.txt'")

Text file saved as 'combined_patient_data.txt'
